<a href="https://colab.research.google.com/github/dipikamishra/my-repository/blob/main/Answers_04_AssociationPatternMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Association Pattern Mining

In this week's practical we will learn and practice the following skills:
- Explore how transaction data are stored
- Use apriori and FP growth algorithms to find frequent item sets
- Explore the relationship between itemset size and frequency
- Generate association rules
- Understand support, confidence, and lift
- Interpret association rules, hypothesize relationships, and propose further work.
- Documenting our work

Apriori builds combinations gradually

Apriori generates candidates;
FP-Growth compresses transactions into a tree.
tiny example:

Transaction Items bought

T1 Bread, Milk
T2 Bread, Diapers, Beer, Eggs
T3 Milk, Diapers, Beer, Cola
T4 Bread, Milk, Diapers, Beer
T5 Bread, Milk, Diapers, Cola
Each row is one transaction. Each transaction contains several items. An itemset is simply a group of items.

Examples:

{Bread} = 1-itemset

{Bread, Milk} = 2-itemset

{Bread, Milk, Diapers} = 3-itemset

FP-Growth stands for Frequent Pattern Growth.

It is an algorithm used to find frequent itemsets in transaction data.

For your animal example, FP-Growth can help find patterns such as:

{Red-rumped Parrot, Whiptail Wallaby}

appearing together frequently across NSW regions.

A simple way to remember it is:

FP = Frequent Pattern Growth = growing/finding larger frequent patterns

Unlike Apriori, FP-Growth does not generate lots of candidate itemsets. It uses an FP-tree (Frequent Pattern Tree), which often makes it faster for larger datasets.

## Preparing your workspace
In this practical we'll be using Python/[Pandas](https://pandas.pydata.org) to explore and visualise some example data using [Jupyter Notebooks](https://jupyter.org) in [Google Colaboratory](https://colab.research.google.com).

At the start of each practical you should make a copy of the notebook in your own google drive:
- File -> Save as copy in Drive

If you would prefer to work with a Jupyter Lab session ony your own machine you can download the notebook directly via:
- File -> Downaload -> Download .ipynb

In this practial we will be preparing data for use with some data mining applications.

The data for this week can be found on [GitHub](https://github.com/PaulHancock/COMP5009_pracs).


## Animals of NSW

Our task for today is to use association pattern mining to understand the co-occorance of animals across NSW.
There are many reasons that animals will be observed in the same regions including:
- They feed on similar plants or animals,
- They prefer the same habitat for breeding, mating, or resting,
- The animals have a relationship with each other such as predator/prey,
- The animals are ubiquitous in NSW and thus can be expected to be found everywhere you look, and their co-occurance is expected by chance alone.

Before we can start to explore the reasons for co-occurance we need to first identify which animals are observed together, and thus we will use assocation pattern mining to find these patterns.

Once we have established patterns of occurance we can generate association rules, which may give some insight as to the reason by the co-occurances.

### The data
The data set that we will be using this week is taken from the [NSW BioNet Atlas](https://doi.org/10.15468/14jd9g).
The data contain records from the NSW Department of planning, industry and environment BioNet Atlas database of flora and fauna sightings.
It includes records from other custodians such as the National Herbarium of NSW, Forests NSW, Australian Bird and Bat Banding Scheme and the Australian Museum.
The full data set is 24GB of detailed sighting information, with 14,861,875 observations (rows) and 183 features (columns).
Many of the features are either empty, not relevant, or have been redacted.

For this work I have transformed the data as follows:
- Select only rows which contain animals ("kingdom" == "animalia")
- Select only rows which contains observations "year"==2010
- Select features "vernacularName" (a.k.a "Name") and "county" (a.k.a "Region")
- The above selection produces 7,441,321 instaces and two features ("Name", "Region")
- There are 2475 unique animal names, and 173 unique regions. A random sample of 10 of the top 800 most common animals are selected, and instances that don't contain these animals are removed.
- The data are then transformed by grouping by "Region" to obtain a list of the unique animals names that were observed.
- Finally, the data are one-hot encoded into a table of regions and features to form 157 rows x 10 features.

The animals that are present in the data set are:
- Australasian Shoveler
- Mountain Brushtail Possum
- Red-rumped Parrot
- Barrabarruun
- Watson's Tree Frog
- Mainland Black-faced Cuckoo-shrike
- Eastern Bent-winged Bat
- Common Planigale
- Black-faced Monarch
- Whiptail Wallaby

A `.csv` of these "transactions" has been made available on [GitHub](https://github.com/PaulHancock/COMP5009_pracs/raw/refs/heads/main/data/animals_transactions_2010.csv)


**References / further reading:**
- NSW Department of planning, industry and environment (2023). NSW BioNet Atlas. Occurrence dataset https://doi.org/10.15468/14jd9g accessed via GBIF.org on 2025-07-17.




In [ ]:
import pandas as pd
import urllib
import urllib.request
import numpy as np

## Load the data file into a pandas data frame

Pands will let us read a file directly from a url.

Once loaded you should determine the data size and dimension, as well as the features that are present, and the data types.

In [ ]:
# Load data
data_url = 'https://github.com/PaulHancock/COMP5009_pracs/raw/refs/heads/main/data/animals_transactions_2010.csv'
df = pd.read_csv(data_url)
df

,Australasian Shoveler,Mountain Brushtail Possum,Red-rumped Parrot,Barrabarruun,Watson's Tree Frog,Mainland Black-faced Cuckoo-shrike,Eastern Bent-winged Bat,Common Planigale,Black-faced Monarch,Whiptail Wallaby
0,True,False,True,False,False,False,False,False,False,False
1,False,False,True,True,False,False,True,False,False,False
2,False,False,False,False,False,False,False,False,True,False
3,True,True,True,True,False,False,True,False,True,True
4,False,True,True,True,False,False,True,True,True,False
...,...,...,...,...,...,...,...,...,...,...
152,True,False,True,True,False,True,True,False,True,False
153,True,True,True,True,False,True,True,False,True,False
154,False,True,True,True,False,False,True,False,True,False
155,True,False,False,False,False,False,False,False,True,False


In [ ]:
# determine the size and dimension of the data
df.shape

# the practical description tells us that after grouping
#and one-hot encoding, the final table has:
#One-hot encoding converts categories/items into separate True/False
#or 0/1 columns so algorithms can process them.

# 157 rows × 10 features

# Here:

# 157 rows = 157 regions
# 10 columns/features = the 10 selected animals

(157, 10)

In [ ]:
# Look for missing data
df.isnull().sum()
#tells you how many missing values are in each column of a pandas DataFrame.

,0
Australasian Shoveler,0
Mountain Brushtail Possum,0
Red-rumped Parrot,0
Barrabarruun,0
Watson's Tree Frog,0
Mainland Black-faced Cuckoo-shrike,0
Eastern Bent-winged Bat,0
Common Planigale,0
Black-faced Monarch,0
Whiptail Wallaby,0


In [ ]:
# identify data types
df.dtypes

# in this dataset, each animal column represents a yes/no condition:

# True = this animal was observed in that region
# False = this animal was not observed in that region

# So bool is exactly the right data type.

,0
Australasian Shoveler,bool
Mountain Brushtail Possum,bool
Red-rumped Parrot,bool
Barrabarruun,bool
Watson's Tree Frog,bool
Mainland Black-faced Cuckoo-shrike,bool
Eastern Bent-winged Bat,bool
Common Planigale,bool
Black-faced Monarch,bool
Whiptail Wallaby,bool


Confirm that the data shape and type is as you exepct, and that there are no missing data.
If there are any issues, you should address them.

This is very suitable for association pattern mining because algorithms like Apriori and FP-Growth essentially want to know:

Is the item present in this transaction or not?

In supermarket terms:

True = customer bought the item

False = customer did not buy the item

In our animal dataset:

True = animal appeared in the region

False = animal did not appear in the region


## Use the Apriori algorithm to find frequent itemsets

Select the Apriori algorithm and perform frequent itemset mining with `minsup = 0.2`.

Determine the number of frequent 2-itemsets, and 3-itemsets.
- The best three rules with largest confidence. Examine these rules and describe them in your own words.

The `apriori` algorithm is found in the `mlxtend` package, so we import it along with the `association_rules` function.

Association rules are if–then patterns that describe how items are related in transaction data.

For example, in supermarket data:

{Bread}→{Milk}

This means:

When Bread is present in a transaction, Milk is also often present.

For our animal dataset, an association rule might be:

{Red-rumped Parrot}→{Whiptail Wallaby}

This means:

In regions where the Red-rumped Parrot is observed, the Whiptail Wallaby is also often observed.

The left side is called the antecedent and the right side is called the consequent:

antecedent
A → B consequent

Association rules are usually evaluated using:

Support — how often A and B occur together overall

Confidence — when A occurs, how often does B also occur?

Lift — do A and B occur together more than we would expect by chance?

or example:

Parrot→Wallaby

with:

support = 0.30

confidence = 0.75

lift = 1.5

would mean 30% of all regions contain both animals, 75% of regions containing the parrot also contain the wallaby, and the wallaby is about 1.5 times more likely to occur when the parrot occurs than it is normally.

The important point is that an association rule shows a relationship, not causation. It does not mean the parrot causes the wallaby to be there.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
ap_itemsets = apriori(df,
                      min_support=0.2,  # choose the (relative) minsup
                      use_colnames=True)

min_support is a threshold you choose to decide how frequent an itemset must be before Apriori keeps it.

min_support = 0.2 means:

Keep only animal combinations that occur in at least 20% of the regions.

our dataset has about 157 regions, so:

0.2×157=31.4

So an itemset needs to occur in roughly 32 or more regions to be kept.

In [ ]:
ap_itemsets

,support,itemsets
0,0.624204,(Australasian Shoveler)
1,0.382166,(Mountain Brushtail Possum)
2,0.878981,(Red-rumped Parrot)
3,0.687898,(Barrabarruun)
4,0.273885,(Mainland Black-faced Cuckoo-shrike)
...,...,...
61,0.292994,"(Eastern Bent-winged Bat, Mountain Brushtail P..."
62,0.312102,"(Eastern Bent-winged Bat, Mountain Brushtail P..."
63,0.394904,"(Eastern Bent-winged Bat, Barrabarruun, Red-ru..."
64,0.235669,"(Australasian Shoveler, Barrabarruun, Red-rump..."


we run Apriori, and it has returned the frequent itemsets together with their support values.

For example, this row:

support     itemsets

0.624204    (Australasian Shoveler)

And a row with multiple animals, such as:

0.394904   (Eastern Bent-winged Bat, Barrabarruun, Red-rumped Parrot)

means that exact combination of 3 animals appears together in about 39.5% of the regions.


Now the next step is to find the size of each itemset.

For example:

(Red-rumped Parrot) → size = 1

(Red-rumped Parrot, Whiptail Wallaby) → size = 2

(Bat, Parrot, Wallaby) → size = 3

That size is usually called k.

Now that we have our itemsets we want to chose those with `2<=k<=3`.
This isn't explicitly stored within our dataframe so we'll make a new column which is just the value of `len(itemsets)`.
------------
We only want itemsets containing either 2 animals or 3 animals.

The problem is that the Apriori output only currently has:

support

itemsets

There is no column that says how many animals are inside each itemset.

So this function is created:

In [ ]:
def find_k(row):
  """Return the number of items in the itemset"""
  #Take one row, look at its itemsets, and count how many items are inside it.
  return len(row['itemsets'])

# Create a new column which counts the number of items in the itemset
ap_itemsets['k'] = ap_itemsets.apply(find_k, # Apply the function `find_k`
                                     axis=1) # apply the function to each row

In [ ]:
ap_itemsets

,support,itemsets,k
0,0.624204,(Australasian Shoveler),1
1,0.382166,(Mountain Brushtail Possum),1
2,0.878981,(Red-rumped Parrot),1
3,0.687898,(Barrabarruun),1
4,0.273885,(Mainland Black-faced Cuckoo-shrike),1
...,...,...,...
61,0.292994,"(Eastern Bent-winged Bat, Mountain Brushtail P...",4
62,0.312102,"(Eastern Bent-winged Bat, Mountain Brushtail P...",4
63,0.394904,"(Eastern Bent-winged Bat, Barrabarruun, Red-ru...",4
64,0.235669,"(Australasian Shoveler, Barrabarruun, Red-rump...",5


In [ ]:
ap_itemsets[ap_itemsets['k']==1].iloc[4]
#Find all frequent itemsets containing exactly one animal, then return the 5th one.
# iloc[0] → first row
# iloc[1] → second row
# iloc[2] → third row
# iloc[3] → fourth row
# iloc[4] → fifth row


,4
support,0.273885
itemsets,(Mainland Black-faced Cuckoo-shrike)
k,1


Us `groupby` and `count` to determine the number of item sets for each size of item set.

In [ ]:
ap_itemsets.groupby('k').count()

# Remember, k is the number of animals in the itemset.

# So if k = 1, that means a 1-animal itemset.
 #If k = 2, that means a pair of animals.
 #If k = 3, that means a group of three animals.

,support,itemsets
k,,
1,7,7
2,20,20
3,24,24
4,13,13
5,2,2


Now use `nlargest` to list the 10 itemsets with the highest support

In [ ]:
# Now lets see the top 10 itemsets
# try either .head() or .nlargest(10,'support')
ap_itemsets.nlargest(10, 'support')

# From the ap_itemsets dataframe, show the 10 rows with the largest support values.

# So it is trying to find the 10 most frequent itemsets.

,support,itemsets,k
2,0.878981,(Red-rumped Parrot),1
3,0.687898,(Barrabarruun),1
5,0.668790,(Eastern Bent-winged Bat),1
17,0.630573,"(Barrabarruun, Red-rumped Parrot)",2
0,0.624204,(Australasian Shoveler),1
19,0.605096,"(Eastern Bent-winged Bat, Red-rumped Parrot)",2
6,0.598726,(Black-faced Monarch),1
8,0.598726,"(Australasian Shoveler, Red-rumped Parrot)",2
22,0.566879,"(Eastern Bent-winged Bat, Barrabarruun)",2
45,0.509554,"(Eastern Bent-winged Bat, Barrabarruun, Red-ru...",3


Are the top 10 itemsets are all 1-itemsets? Is this surprising to you?

If there are any 2 or 3 itemsets in the top 10, identify the animals and see if you can see any comonality between them, in terms of habitat requirements, diet, range, etc.
Make an initial hypothesis about what is driving the co-occurance of these animals.

No, the top 10 itemsets are not all 1-itemsets.

From table, the top 10 contain:

5 one-itemsets
4 two-itemsets
1 three-itemset

The 2-itemsets are:

Barrabarruun + Red-rumped Parrot
Eastern Bent-winged Bat + Red-rumped Parrot
Australasian Shoveler + Red-rumped Parrot
Eastern Bent-winged Bat + Barrabarruun

The 3-itemset is:

Eastern Bent-winged Bat + Barrabarruun + Red-rumped Parrot

The strongest 2-itemset in your output is:

{Barrabarruun, Red-rumped Parrot}

with support about:

0.6306=63.1%

So these two animals are observed together in about 63% of the regions in this dataset.

That is fairly high, so it is worth asking why.

Several pairs and one group of three animals have very high support. One possible explanation is that some of these species have broad and overlapping distributions across NSW and can occur in common woodland, grassland, forest or modified environments. Therefore, their co-occurrence may be driven partly by overlapping habitat and geographic range rather than a direct interaction between the species. Further investigation using habitat, diet and geographic data would be needed to determine the cause.

Support only tells us:

How often are these animals together?

Association rules, especially confidence and lift, help us ask:

Is one animal particularly informative about the presence of another, or are they together simply because both are common?

## Turn item-sets into association rules

Now that we have identified frequent item-sets we can start to look at the links between them using association rules.

we can make two possible association rules:

Parrot → Bat means:

In regions where the Parrot is observed, how often is the Bat also observed?

Bat → Parrot means:

In regions where the Bat is observed, how often is the Parrot also observed?

We use these itemsets to generate association rules with a minimum confidence of 0.8.

In [ ]:
ap_rules = association_rules(ap_itemsets,
                             metric='confidence',
                             min_threshold=0.8) # choose the minimum confidence value

# “Apriori tells us which animals frequently occur together.
# Association rules now tell us whether the presence of one animal is strongly associated with the presence of another.
# We are keeping only rules with at least 80% confidence.

In [ ]:
ap_rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Australasian Shoveler),(Red-rumped Parrot),0.624204,0.878981,0.598726,0.959184,1.091245,1.0,0.050063,2.964968,0.222503,0.661972,0.662728,0.820172
1,(Mountain Brushtail Possum),(Red-rumped Parrot),0.382166,0.878981,0.350318,0.916667,1.042874,1.0,0.014402,1.452229,0.066542,0.384615,0.311404,0.657609
2,(Mountain Brushtail Possum),(Barrabarruun),0.382166,0.687898,0.343949,0.900000,1.308333,1.0,0.081058,3.121019,0.381443,0.473684,0.679592,0.700000
3,(Mountain Brushtail Possum),(Eastern Bent-winged Bat),0.382166,0.668790,0.363057,0.950000,1.420476,1.0,0.107469,6.624204,0.479110,0.527778,0.849038,0.746429
4,(Mountain Brushtail Possum),(Black-faced Monarch),0.382166,0.598726,0.337580,0.883333,1.475355,1.0,0.108767,3.439490,0.521494,0.524752,0.709259,0.723582


The above table shows a range of different measures for the various rules. Of particular interest to us are the following columns:
- Antecedences and Consequents as they are the A,C for Conf(A->C)
- The support of the antecedent and consequent
- Suppot for the item set of A union C
- Confidence and lift of the association rule

Note that the rules above are not sorted by confidence. We should do that ourselves by using the `sort_values` function.

In [ ]:
ap_rules.sort_values('confidence', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
18,"(Mainland Black-faced Cuckoo-shrike, Australas...",(Red-rumped Parrot),0.210191,0.878981,0.210191,1.000000,1.137681,1.0,0.025437,inf,0.153226,0.239130,1.000000,0.619565
6,(Mainland Black-faced Cuckoo-shrike),(Red-rumped Parrot),0.273885,0.878981,0.267516,0.976744,1.111223,1.0,0.026776,5.203822,0.137845,0.302158,0.807834,0.640546
40,"(Mainland Black-faced Cuckoo-shrike, Barrabarr...",(Red-rumped Parrot),0.222930,0.878981,0.216561,0.971429,1.105176,1.0,0.020609,4.235669,0.122469,0.244604,0.763910,0.608903
48,"(Mainland Black-faced Cuckoo-shrike, Black-fac...",(Red-rumped Parrot),0.216561,0.878981,0.210191,0.970588,1.104220,1.0,0.019839,4.114650,0.120473,0.237410,0.756966,0.604859
47,"(Mainland Black-faced Cuckoo-shrike, Eastern B...",(Red-rumped Parrot),0.210191,0.878981,0.203822,0.969697,1.103206,1.0,0.019068,3.993631,0.118448,0.230216,0.749601,0.600791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,"(Mainland Black-faced Cuckoo-shrike, Red-rumpe...",(Barrabarruun),0.267516,0.687898,0.216561,0.809524,1.176808,1.0,0.032537,1.638535,0.205115,0.293103,0.389699,0.562169
44,"(Barrabarruun, Red-rumped Parrot)",(Eastern Bent-winged Bat),0.630573,0.668790,0.509554,0.808081,1.208273,1.0,0.087833,1.725779,0.466595,0.645161,0.420552,0.784993
90,"(Mountain Brushtail Possum, Eastern Bent-winge...","(Red-rumped Parrot, Black-faced Monarch)",0.363057,0.477707,0.292994,0.807018,1.689357,1.0,0.119559,2.706427,0.640652,0.534884,0.630509,0.710175
103,"(Eastern Bent-winged Bat, Black-faced Monarch)","(Barrabarruun, Red-rumped Parrot)",0.490446,0.630573,0.394904,0.805195,1.276925,1.0,0.085642,1.896391,0.425605,0.543860,0.472682,0.715729


Describe the first three that you see above in your own words.

The [Eastern Bent-winged Bat](https://www.nationalparks.nsw.gov.au/plants-and-animals/eastern-bentwing-bat) is fun little fellow, but hard to find, due to their small size and nocturnal nature.
If we were to go looking for this bat, what other animals could we use as a proxy to tell that we were looking in the right place?

That is - if B is the Eastern Bent-winged Bat, what animal(s) A would give a rule Conf(A -> B) with high confidence and lift?

In [ ]:
# choose all the rules wihch have our bat as the consquent
bat_rules = ap_rules[ap_rules.consequents == frozenset(['Eastern Bent-winged Bat'])]

# choose the three rules with the highest confidence
bat_rules.sort_values('confidence',
                        ascending=False).head(3) # choose the top 3 only

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
32,"(Mountain Brushtail Possum, Barrabarruun)",(Eastern Bent-winged Bat),0.343949,0.66879,0.331210,0.962963,1.439859,1.0,0.101181,8.942675,0.465646,0.485981,0.888177,0.729101
38,"(Mountain Brushtail Possum, Black-faced Monarch)",(Eastern Bent-winged Bat),0.337580,0.66879,0.324841,0.962264,1.438814,1.0,0.099071,8.777070,0.460407,0.476636,0.886067,0.723989
95,"(Mountain Brushtail Possum, Barrabarruun, Blac...",(Eastern Bent-winged Bat),0.324841,0.66879,0.312102,0.960784,1.436601,1.0,0.094852,8.445860,0.450135,0.457944,0.881599,0.713725


## Use the FP-Growth algorithm

Use the FP-Growth algorithm to generate item sets and compare them to those found by the Apriori algorithm.

What are the main differences that you see between the results of the two algorithms?

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

In [ ]:
fp_itemsets = fpgrowth(df,
                       min_support=0.2, # Same as before
                       use_colnames=True)

# add our label for the size of the itemset
fp_itemsets['k'] = fp_itemsets.apply(find_k, axis=1)

fp_itemsets.head(10)

,support,itemsets,k
0,0.878981,(Red-rumped Parrot),1
1,0.624204,(Australasian Shoveler),1
2,0.687898,(Barrabarruun),1
3,0.668790,(Eastern Bent-winged Bat),1
4,0.598726,(Black-faced Monarch),1
5,0.382166,(Mountain Brushtail Possum),1
6,0.273885,(Mainland Black-faced Cuckoo-shrike),1
7,0.598726,"(Australasian Shoveler, Red-rumped Parrot)",2
8,0.420382,"(Australasian Shoveler, Barrabarruun)",2
9,0.414013,"(Eastern Bent-winged Bat, Australasian Shoveler)",2


In [ ]:
ap_itemsets.head(10)

,support,itemsets,k
0,0.624204,(Australasian Shoveler),1
1,0.382166,(Mountain Brushtail Possum),1
2,0.878981,(Red-rumped Parrot),1
3,0.687898,(Barrabarruun),1
4,0.273885,(Mainland Black-faced Cuckoo-shrike),1
5,0.668790,(Eastern Bent-winged Bat),1
6,0.598726,(Black-faced Monarch),1
7,0.248408,"(Mountain Brushtail Possum, Australasian Shove...",2
8,0.598726,"(Australasian Shoveler, Red-rumped Parrot)",2
9,0.420382,"(Australasian Shoveler, Barrabarruun)",2


## Wrap up

Summarise your work for today.
Include a description of:
- any data cleaning activities that you engaged in,
- the task that was to be completed,
- any differences that you saw between the two algorithms that we used.

Consider now that the data set contained all 2475 animals over 173 regions and all years of study.
What are some of the issues that you might run into when working with this data set?
Consider issues related to the practical processinng of data, as well as issues related to the interpretation of results.
